In [1]:
from arch import arch_model

print("ARCH installed successfully")

ARCH installed successfully


In [2]:
import pandas as pd
import numpy as np

from arch import arch_model

df = pd.read_csv(
    "../data/processed/model_dataset.csv",
    index_col=0,
    parse_dates=True
)

df.head()

,Target_GK_Vol,Target_Vol_t1,AAPL_Return,Ret_Lag1,Ret_Lag5,Ret_Lag10,Vol_Roll5,Vol_Roll20,Vol_Roll60,Vol_Change,VIX_Close,VIX_Change,SPY_Return,SPY_Vol,MSFT_Return,MSFT_Vol,AAPL_Volume,Volume_Change
Date,,,,,,,,,,,,,,,,,,
2016-03-30,1.054690,0.512436,1.730836,2.339564,0.761877,1.989449,0.867093,0.975008,1.343805,-0.025540,13.56,-1.899248,0.437818,0.430684,0.619527,0.936978,182404400,0.379831
2016-03-31,0.512436,0.942546,-0.521597,1.730836,-0.554369,1.320394,0.819679,0.957179,1.325470,-0.018286,13.95,2.835518,-0.243004,0.351740,0.326437,0.879715,103553600,-0.566137
2016-04-01,0.942546,1.155689,0.913326,-0.521597,-0.434400,-0.160564,0.827529,0.970517,1.316621,0.013935,13.10,-6.286724,0.678862,0.760426,0.613720,1.200247,103496000,-0.000556
2016-04-04,1.155689,0.824183,1.022134,0.913326,-0.455278,0.113359,0.941198,0.948571,1.306780,-0.022612,14.12,7.497996,-0.324290,0.364468,-0.252242,0.843477,149424800,0.367260
2016-04-05,0.824183,1.067183,-1.185913,1.022134,2.339564,-0.009415,0.897909,0.926853,1.282988,-0.022896,15.42,8.807315,-1.003826,0.804057,-1.582015,0.813800,106314800,-0.340389


In [3]:
returns = df["AAPL_Return"]

returns = returns.dropna()

In [4]:
train_window = 1250

train_returns = returns.iloc[:train_window]

test_returns = returns.iloc[train_window:]

In [5]:
garch = arch_model(
    train_returns,
    mean="Zero",
    vol="GARCH",
    p=1,
    q=1
)

garch_fit = garch.fit(
    disp="off"
)

print(
    garch_fit.summary()
)

                       Zero Mean - GARCH Model Results                        
Dep. Variable:            AAPL_Return   R-squared:                       0.000
Mean Model:                 Zero Mean   Adj. R-squared:                  0.001
Vol Model:                      GARCH   Log-Likelihood:               -2405.27
Distribution:                  Normal   AIC:                           4816.53
Method:            Maximum Likelihood   BIC:                           4831.92
                                        No. Observations:                 1250
Date:                Wed, Jun 17 2026   Df Residuals:                     1250
Time:                        11:33:04   Df Model:                            0
                             Volatility Model                             
                 coef    std err          t      P>|t|    95.0% Conf. Int.
--------------------------------------------------------------------------
omega          0.1888  6.394e-02      2.952  3.154e-03 [6.346e-0

In [6]:
garch_forecasts = []

for i in range(len(test_returns)):

    train_slice = returns.iloc[:train_window + i]

    model = arch_model(
        train_slice,
        mean="Zero",
        vol="GARCH",
        p=1,
        q=1
    )

    fit = model.fit(
        disp="off"
    )

    forecast = fit.forecast(
        horizon=1
    )

    pred_vol = np.sqrt(
        forecast.variance.iloc[-1,0]
    )

    garch_forecasts.append(
        pred_vol
    )

In [7]:
results_df = pd.DataFrame(
    {
        "Actual":
        df["Target_Vol_t1"].iloc[train_window:],

        "GARCH_Forecast":
        garch_forecasts
    }
)

results_df = results_df.dropna()

In [8]:
def calculate_metrics(
    actual,
    predicted
):

    epsilon = 1e-10

    mse = np.mean(
        (actual - predicted) ** 2
    )

    rmse = np.sqrt(mse)

    mae = np.mean(
        np.abs(actual - predicted)
    )

    actual_var = np.maximum(
        actual**2,
        epsilon
    )

    predicted_var = np.maximum(
        predicted**2,
        epsilon
    )

    qlike = np.mean(
        actual_var/predicted_var
        - np.log(
            actual_var/predicted_var
        )
        - 1
    )

    return {
        "QLIKE": qlike,
        "RMSE": rmse,
        "MAE": mae
    }

In [9]:
garch_metrics = calculate_metrics(
    results_df["Actual"],
    results_df["GARCH_Forecast"]
)

print(garch_metrics)

{'QLIKE': np.float64(0.39867979119541896), 'RMSE': np.float64(0.7552148563199561), 'MAE': np.float64(0.6036260432953566)}


In [10]:
egarch = arch_model(
    train_returns,
    mean="Zero",
    vol="EGARCH",
    p=1,
    q=1
)

In [13]:
comparison = pd.DataFrame(
    [
        {
            "Model": "Historical Average",
            "QLIKE": 0.3385,
            "RMSE": 0.5819,
            "MAE": 0.3927
        },

        {
            "Model": "GARCH(1,1)",
            **garch_metrics
        }

        # Uncomment after EGARCH runs
        # {
        #     "Model": "EGARCH(1,1)",
        #     **egarch_metrics
        # }
    ]
)

comparison

,Model,QLIKE,RMSE,MAE
0,Historical Average,0.33850,0.581900,0.392700
1,"GARCH(1,1)",0.39868,0.755215,0.603626
